In [ ]:
# --timeframe 1d   : Таймфрейм свечей (дневные данные).
# --start-year 2000: Глубина загрузки истории (начиная с 2000 года).
# --workers 6      : Количество параллельных потоков для ускорения загрузки.

!python -m _tools.update_market_data --timeframe 1d --start-year 2000 --workers 6
!python -m _tools.update_macro --timeframe 1d --start-year 2000 --workers 6
# Выполняет комплексную проверку целостности, отсутствия пропусков и корректности OHLCV данных.
!python -m _tools.check_data_quality

In [ ]:
# --timeframe 1d       : Интервал данных — дневные свечи.
# --lookback 60        : Глубина истории — модель смотрит на 60 дней назад.
# --horizon 10         : Горизонт прогноза — ищем выход по барьерам в течение 10 дней.
# --auto               : Режим автоматического расчета уровней TP/SL на основе волатильности.
# --percentile 75      : Перцентиль волатильности для отсечения аномальных выбросов при авто-разметке.
# --init_split         : Дата начала первого разделения данных на Train и Val.
# --val_interval 2     : Продолжительность валидационного периода в годах.
# --split_interval 2   : Шаг смещения окна Walk-Forward в годах.
# --endpoint           : Дата окончания формирования всех временных интервалов.
# --corr_threshold     : Порог удаления коррелирующих признаков (убираем дубликаты > 85%).
# --cum_threshold      : Порог кумулятивной важности (оставляем топ фичей, дающих 99% влияния).
# --force              : Раскомментируйте параметр ниже для полной перезаписи кэшированных данных.

!python -m _tools.init_dataset \
    --timeframe 1d \
    --lookback 60 \
    --horizon 10 \
    --auto \
    --percentile 75 \
    --init_split 2010-01-01 \
    --val_interval 2 \
    --split_interval 2 \
    --endpoint 2024-01-01 \
    --corr_threshold 0.85 \
    --cum_threshold 0.99 \
    #--force

In [ ]:
#полная проверка подготовленных на предыдущем этапе данных
!python -m _tools.verify_data

In [ ]:
import os
import subprocess
from pathlib import Path

DATASET_DIR = Path("data/processed/2000_2026_1d")
print("🚀 Запуск массового обучения моделей (Walk-Forward)...")

# Получаем список папок fold_
folds = sorted([d for d in DATASET_DIR.glob("fold_*") if d.is_dir()])

try:
    for fold_dir in folds:
        fold_name = fold_dir.name
        
        # --- ПРОВЕРКА УДАЛЕНА ---
        # Мы убрали 'if list(models_dir.glob("*.keras")): continue'
        # Теперь внешний скрипт всегда вызывает train_model, 
        # а тот уже сам читает флаг --append и добавляет модели!
        
        print("="*60)
        print(f"🔥 Обучение (добавление) нейросети для: {fold_name}")
        print("="*60)
        
        # Формируем команду вызова
        cmd = [
            "python", "-m", "_tools.train_ltsm_model",
            "--dataset_dir", str(DATASET_DIR),
            "--fold", fold_name,
            "--runs", "50",
            "--batch_size", "8192", 
            "--epochs", "50",
            "--l2_reg", "1e-4",
            "--lr", "1e-3",
            "--append"  # <--- Теперь этот флаг дойдет до адресата!
        ]
        
        # Запускаем процесс и позволяем ему выводить логи в реальном времени
        process = subprocess.Popen(cmd)
        
        # Ждем завершения, но позволяем Jupyter перехватить прерывание
        process.wait()
        
        print(f"✅ [{fold_name}] Завершен!")

except KeyboardInterrupt:
    print("\n🛑 Остановка пайплайна пользователем!")
    if 'process' in locals():
        process.terminate() # Мягкая остановка текущего процесса
        print("⏳ Завершаем текущий фолд...")
except Exception as e:
    print(f"❌ Ошибка: {e}")

print("🎉 Процесс полностью остановлен.")

🚀 Запуск массового обучения моделей (Walk-Forward)...
🔥 Обучение (добавление) нейросети для: fold_2010


In [ ]:
#очистка наименне успешных ltsm моделей (остается топ 3)
!python -m _tools.clean_lstm_models

In [6]:
!python -m _tools.evaluate_lstm_predictions

📊 Загрузка данных из data/processed/2000_2026_1d/rl_env/environment_data.parquet...
⚙️ Расчет сигналов и доходностей...
🔍 Анализ 63 тикеров...

🏆 ТОП-5 ТИКЕРОВ ПО ПРЕВОСХОДСТВУ НАД РЫНКОМ (ALPHA)
   Ticker  Win_Rate_%  Strat_Return_%  B&H_Return_%  Alpha_%
RUAL@MISX       53.93         1360.09         37.17  1322.92
AFKS@MISX       51.08          931.93        -77.19  1009.12
VTBR@MISX       52.25          471.33        -79.01   550.34
SNGS@MISX       53.17          451.22        -23.95   475.18
GAZP@MISX       55.12          436.38        -28.03   464.41

📉 ХУДШИЕ 5 ТИКЕРОВ ПО ПРЕВОСХОДСТВУ НАД РЫНКОМ (ALPHA)
    Ticker  Win_Rate_%  Strat_Return_%  B&H_Return_%  Alpha_%
 YDEX@MISX       45.87          -42.49        341.35  -383.84
 NVTK@MISX       49.46          -23.19        521.13  -544.33
 PHOR@MISX       49.56          -50.85        596.36  -647.21
LSNGP@MISX       46.96          -77.38        743.67  -821.05
 PLZL@MISX       48.91           56.64       1475.16 -1418.52

📈 ОБЩАЯ С

In [ ]:
!python -m _tools.prepare_rl_env

In [ ]:
!python -m _tools.train_rllib_pbt --population 4 --iterations 3000